# Tutorial: additive and zero-sum sequences

`zero_sum_sequences` represents finite multisets over additive parents and computes zero-sum atoms, factorization length sets, witnesses, and factorization graphs.  This tutorial introduces the public API through small examples over $C_3$ whose results can be checked by hand.

The notebook runs in an ordinary Python environment or directly on Binder.  The assertions embedded in the examples provide executable checks of the displayed results.

## 1. Configure a sequence space

An `AdditiveSequenceSpace` fixes two pieces of data once: an additive parent and an upper bound for its Davenport constant.  Calling the space constructs immutable finite multisets of elements of that parent.  The package never tries to infer this bound, so a bound that is too small can omit atoms and make factorization results incomplete.

For the cyclic group $C_3=\mathbb Z/3\mathbb Z$, the Davenport constant is 3.

In [ ]:
from zero_sum_sequences import (
    AdditiveSequenceSpace,
    AtomCatalogue,
    FactorizationSolver,
    FiniteAdditiveGroup,
)

modulo_three = FiniteAdditiveGroup(
    range(3),
    zero=0,
    add=lambda left, right: (left + right) % 3,
    coerce=lambda value: int(value) % 3,
)
C3 = AdditiveSequenceSpace(modulo_three, davenport_bound=3)
print('base parent:', C3.base_parent)
print('Davenport bound:', C3.davenport_bound)

## 2. Construct and inspect sequences

The constructor coerces terms through the configured parent and stores a canonical multiplicity table.  Thus the order used to construct a sequence is irrelevant.  A sequence is *zero-sum* when the sum of all its terms is zero, and it is an *atom* when it is a nonempty zero-sum sequence without a proper nonempty zero-sum subsequence.

In [ ]:
sample = C3([2, 1, 2, 1])

print('sequence:', sample)
print('parent is C3:', sample.parent() is C3)
print('terms:', tuple(sample))
print('length:', len(sample))
print('support:', sample.support)
print('multiplicities:', sample.multiplicities)
print('total:', sample.total())
print('zero-sum:', sample.is_zero_sum())
print('atom:', sample.is_atom())

assert tuple(sample) == (C3.base_parent(1), C3.base_parent(1), C3.base_parent(2), C3.base_parent(2))
assert len(sample) == 4
assert sample.total() == C3.base_parent.zero()
assert sample.is_zero_sum()
assert not sample.is_atom()  # 1 · 2 is already a zero-sum subsequence

## 3. Multiset arithmetic

Addition combines multiplicities, subtraction removes a subsequence, and multiplication by a nonnegative integer repeats a sequence.  `divides` tests the multiset-subsequence relation.  All of these operations keep the same configured sequence space.

In [ ]:
pair = C3([1, 2])
triple_pair = 3 * pair

print('pair:', pair)
print('pair + pair:', pair + pair)
print('sample - pair:', sample - pair)
print('3 * pair:', triple_pair)
print('pair divides sample:', pair.divides(sample))

assert pair + pair == sample
assert sample - pair == pair
assert triple_pair == C3([1, 1, 1, 2, 2, 2])
assert pair.divides(sample)
assert not triple_pair.divides(sample)

## 4. Factorizations and length sets

Consider $S=1^3 2^3$ over $C_3$.  It has two visible factorizations into atoms:

$$S=(1^3)(2^3)=(1\cdot2)^3.$$

Their lengths are 2 and 3.  `length_set()` computes the complete set, while `factorization_witnesses()` keeps one deterministic witness for each length.

In [ ]:
sequence = C3([1, 1, 1, 2, 2, 2])
one_cubed = C3([1, 1, 1])
two_cubed = C3([2, 2, 2])
mixed_pair = C3([1, 2])

assert all(atom.is_atom() for atom in (one_cubed, two_cubed, mixed_pair))
assert one_cubed + two_cubed == sequence
assert 3 * mixed_pair == sequence

lengths = sequence.length_set()
witnesses = sequence.factorization_witnesses()

def format_factorization(factors):
    return '  *  '.join(str(factor) for factor in factors) or '1'

print('length set:', sorted(lengths))
for length, factors in witnesses.items():
    print(f'length {length}: {format_factorization(factors)}')

assert lengths == {2, 3}
assert witnesses == sequence.factorization_witnesses()
assert {len(factors) for factors in witnesses.values()} == {2, 3}
assert all(sum(factors, C3()) == sequence for factors in witnesses.values())

`factorizations()` enumerates every unordered factorization exactly once.  Enumeration is output-sensitive, so for large examples the length set or one witness per length is usually the more useful query.  Here the complete list is short enough to inspect directly.

In [ ]:
all_factorizations = sorted(sequence.factorizations(), key=len)
for factors in all_factorizations:
    print(f'length {len(factors)}: {format_factorization(factors)}')

assert len(all_factorizations) == 2
assert {len(factors) for factors in all_factorizations} == {2, 3}
assert all(sum(factors, C3()) == sequence for factors in all_factorizations)

## 5. Advanced: reuse a factorization solver

The sequence methods are the ordinary interface.  When several queries will use the same input, `FactorizationSolver` exposes the memoized remainder DAG directly, so the discovered atoms and states are reused.  Its statistics report the number of relevant atom divisors, remainder states, and transitions.

In [ ]:
solver = FactorizationSolver(sequence)
assert solver.length_set() == lengths
assert solver.factorization_witnesses() == witnesses

stats = solver.statistics
print('candidate atoms:', stats.candidate_atoms)
print('remainder states:', stats.states)
print('transitions:', stats.transitions)

assert (stats.candidate_atoms, stats.states, stats.transitions) == (3, 5, 5)

## 6. Inspect the factorization DAG

`sequence.factorization_digraph()` returns the memoized directed acyclic graph.  Its vertices are remainder sequences; an edge removes the atom carried by its label.  A path from the input sequence to the empty sequence therefore encodes a factorization.

In [ ]:
import networkx as nx

dag = sequence.factorization_digraph()
labeled_edges = tuple(dag.edges(data='atom'))
edges = sorted(
    (str(source), str(target), str(atom))
    for source, target, atom in labeled_edges
)

print(f'vertices: {dag.number_of_nodes()}; edges: {dag.number_of_edges()}')
for source, target, atom in edges:
    print(f'{source}  -- remove {atom} -->  {target}')

assert dag.is_directed() and nx.is_directed_acyclic_graph(dag)
assert sequence in dag and C3() in dag
assert {
    len(path) - 1 for path in nx.all_simple_paths(dag, sequence, C3())
} == lengths
assert all(source - atom == target for source, target, atom in labeled_edges)

In [ ]:
# A compact, headless-safe NetworkX plot.
nx.draw_networkx(
    dag,
    pos=nx.spring_layout(dag, seed=7),
    with_labels=False,
    node_size=500,
)

## 7. Enumerate all reduced atoms of a finite group

For a finite group $G$, every atom has length at most its Davenport constant $D(G)$.  In the reduced convention used for factorizations here, the identity is omitted, so every remaining atom has length at least 2.  `enumerate_atom_catalogue()` tests the unordered multisets of nonzero elements with lengths from 2 through the configured bound and returns the atoms in an indexed `AtomCatalogue`.  Internally, combinations with replacement avoid testing different orderings of the same sequence.

For $C_2\oplus C_4$, the exact Davenport constant is $1+(2-1)+(4-1)=5$.  The complete enumeration is still small enough to perform directly.

In [ ]:
from collections import Counter
from itertools import product

group = FiniteAdditiveGroup(
    product(range(2), range(4)),
    zero=(0, 0),
    add=lambda left, right: (
        (left[0] + right[0]) % 2,
        (left[1] + right[1]) % 4,
    ),
    coerce=tuple,
)
C2xC4 = AdditiveSequenceSpace(group, davenport_bound=5)
c2_c4_catalogue = C2xC4.enumerate_atom_catalogue()
atoms_by_length = Counter(map(len, c2_c4_catalogue))

print('total reduced atoms:', len(c2_c4_catalogue))
for length, count in sorted(atoms_by_length.items()):
    print(f'length {length}: {count}')

assert len(c2_c4_catalogue) == 38
assert atoms_by_length == Counter({2: 5, 3: 9, 4: 16, 5: 8})
assert all(atom.is_atom() for atom in c2_c4_catalogue)
assert all(group.zero() not in atom for atom in c2_c4_catalogue)

## 8. Reuse a complete atom catalogue

An `AtomCatalogue` indexes a precomputed atom collection so it can be reused for divisor and factorization queries.  For the earlier $C_3$ input, the complete list of atom divisors is $1^3$, $2^3$, and $1\cdot2$.  **Completeness is the caller's responsibility:** for complete factorization results, the catalogue must contain every atom divisor relevant to the sequence.

In [ ]:
relevant_atoms = (one_cubed, two_cubed, mixed_pair)
catalogue = AtomCatalogue(C3, relevant_atoms)
catalogue_divisors = tuple(catalogue.divisors(sequence))

print('catalogue atoms:', [str(atom) for atom in catalogue])
print('divisors of S:', [str(atom) for atom in catalogue_divisors])

assert set(catalogue_divisors) == set(relevant_atoms)
assert sequence.length_set(atom_catalogue=catalogue) == lengths
assert sequence.factorization_witnesses(atom_catalogue=catalogue) == witnesses
assert FactorizationSolver(sequence, atom_catalogue=catalogue).statistics == stats